# 🎬 IMDb Movie Analysis
**Exploratory Data Analysis on IMDb Top 1000 Movies Dataset**

This notebook covers:
- Data Loading & Cleaning
- Genre Analysis
- Rating Trends Over Decades
- Budget vs Revenue Correlation
- Top Directors & Actors
- Visualizations with Matplotlib & Seaborn

## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (run once)
!pip install pandas numpy matplotlib seaborn plotly kaggle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ All libraries imported successfully!')

## 📥 Step 2: Load Dataset

**Option A – Download from Kaggle (recommended):**
```bash
kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
unzip imdb-dataset-of-top-1000-movies-and-tv-shows.zip
```

**Option B – Direct URL (used below):**

In [ ]:
# Load dataset directly from GitHub mirror (no Kaggle account needed)
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/imdb_top_1000.csv'

try:
    df = pd.read_csv(url)
    print(f'✅ Dataset loaded! Shape: {df.shape}')
except Exception:
    # Fallback: load local file
    df = pd.read_csv('imdb_top_1000.csv')
    print(f'✅ Local dataset loaded! Shape: {df.shape}')

df.head()

## 🔍 Step 3: Data Overview

In [ ]:
print('=== Dataset Info ===')
print(df.info())
print('\n=== Basic Statistics ===')
df.describe()

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

## 🧹 Step 4: Data Cleaning

In [ ]:
# Clean column names
df.columns = df.columns.str.strip()

# Convert Released_Year to numeric
df['Released_Year'] = pd.to_numeric(df['Released_Year'], errors='coerce')

# Clean Gross column (remove commas, convert to numeric)
if 'Gross' in df.columns:
    df['Gross'] = df['Gross'].astype(str).str.replace(',', '').str.replace('$', '')
    df['Gross'] = pd.to_numeric(df['Gross'], errors='coerce')

# Clean Runtime
if 'Runtime' in df.columns:
    df['Runtime'] = df['Runtime'].astype(str).str.replace(' min', '').str.strip()
    df['Runtime'] = pd.to_numeric(df['Runtime'], errors='coerce')

# Clean Meta_score
df['Meta_score'] = pd.to_numeric(df['Meta_score'], errors='coerce')

# Create Decade column
df['Decade'] = (df['Released_Year'] // 10 * 10).astype('Int64').astype(str) + 's'

print('✅ Data cleaned!')
df[['Series_Title', 'Released_Year', 'Decade', 'IMDB_Rating', 'Gross', 'Runtime']].head(10)

## 📊 Step 5: Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IMDb Rating Distribution
axes[0].hist(df['IMDB_Rating'].dropna(), bins=20, color='#f5c518', edgecolor='black')
axes[0].set_title('IMDb Rating Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('IMDb Rating')
axes[0].set_ylabel('Number of Movies')
axes[0].axvline(df['IMDB_Rating'].mean(), color='red', linestyle='--', label=f'Mean: {df["IMDB_Rating"].mean():.2f}')
axes[0].legend()

# Meta Score Distribution
axes[1].hist(df['Meta_score'].dropna(), bins=20, color='#1f77b4', edgecolor='black')
axes[1].set_title('Metacritic Score Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Meta Score')
axes[1].set_ylabel('Number of Movies')
axes[1].axvline(df['Meta_score'].mean(), color='red', linestyle='--', label=f'Mean: {df["Meta_score"].mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.suptitle('Movie Ratings Overview', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('rating_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 🎭 Step 6: Genre Analysis

In [ ]:
# Explode genres (each movie may have multiple genres)
genre_df = df[['Series_Title', 'Genre', 'IMDB_Rating']].copy()
genre_df['Genre'] = genre_df['Genre'].str.split(', ')
genre_df = genre_df.explode('Genre')

# Count movies per genre
genre_count = genre_df['Genre'].value_counts().head(15).reset_index()
genre_count.columns = ['Genre', 'Count']

# Plot
plt.figure(figsize=(14, 6))
bars = plt.barh(genre_count['Genre'], genre_count['Count'], color=sns.color_palette('viridis', 15))
plt.xlabel('Number of Movies', fontsize=12)
plt.title('🎭 Top 15 Movie Genres in IMDb Top 1000', fontsize=15, fontweight='bold')
plt.gca().invert_yaxis()

for bar, val in zip(bars, genre_count['Count']):
    plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('genre_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Average IMDb Rating per Genre
genre_rating = genre_df.groupby('Genre')['IMDB_Rating'].mean().sort_values(ascending=False).head(12).reset_index()
genre_rating.columns = ['Genre', 'Avg_Rating']

plt.figure(figsize=(12, 5))
sns.barplot(data=genre_rating, x='Genre', y='Avg_Rating', palette='rocket')
plt.title('⭐ Average IMDb Rating by Genre', fontsize=14, fontweight='bold')
plt.xlabel('Genre')
plt.ylabel('Average IMDb Rating')
plt.xticks(rotation=45)
plt.ylim(7.5, 9.0)
plt.tight_layout()
plt.savefig('genre_rating.png', bbox_inches='tight', dpi=150)
plt.show()

## 📅 Step 7: Rating Trends Over Decades

In [ ]:
decade_trend = df.groupby('Decade').agg(
    Avg_Rating=('IMDB_Rating', 'mean'),
    Movie_Count=('Series_Title', 'count')
).reset_index().sort_values('Decade')

fig, ax1 = plt.subplots(figsize=(13, 6))

color1 = '#f5c518'
ax1.plot(decade_trend['Decade'], decade_trend['Avg_Rating'],
         marker='o', color=color1, linewidth=2.5, markersize=8, label='Avg IMDb Rating')
ax1.set_xlabel('Decade', fontsize=12)
ax1.set_ylabel('Average IMDb Rating', color=color1, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(7.0, 9.5)

ax2 = ax1.twinx()
ax2.bar(decade_trend['Decade'], decade_trend['Movie_Count'],
        alpha=0.3, color='#1f77b4', label='Movie Count')
ax2.set_ylabel('Number of Movies', color='#1f77b4', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#1f77b4')

plt.title('📅 IMDb Ratings & Movie Count by Decade', fontsize=15, fontweight='bold')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('decade_trend.png', bbox_inches='tight', dpi=150)
plt.show()

## 💰 Step 8: Gross Revenue Analysis

In [ ]:
# Top 10 Highest Grossing Movies
top_gross = df[['Series_Title', 'Gross', 'IMDB_Rating']].dropna().sort_values('Gross', ascending=False).head(10)

plt.figure(figsize=(13, 6))
bars = plt.barh(top_gross['Series_Title'], top_gross['Gross'] / 1e6,
                color=sns.color_palette('plasma', 10))
plt.xlabel('Gross Revenue (Millions $)', fontsize=12)
plt.title('💰 Top 10 Highest Grossing Movies', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

for bar, val in zip(bars, top_gross['Gross']):
    plt.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
             f'${val/1e6:.0f}M', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('top_gross.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# IMDb Rating vs Gross Revenue Scatter
gross_df = df[['Series_Title', 'IMDB_Rating', 'Gross', 'Genre']].dropna()

plt.figure(figsize=(12, 6))
plt.scatter(gross_df['IMDB_Rating'], gross_df['Gross'] / 1e6,
            alpha=0.6, color='#f5c518', edgecolors='grey', s=60)

# Trendline
z = np.polyfit(gross_df['IMDB_Rating'], gross_df['Gross'] / 1e6, 1)
p = np.poly1d(z)
x_line = np.linspace(gross_df['IMDB_Rating'].min(), gross_df['IMDB_Rating'].max(), 100)
plt.plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend Line')

plt.xlabel('IMDb Rating', fontsize=12)
plt.ylabel('Gross Revenue (Millions $)', fontsize=12)
plt.title('⭐ IMDb Rating vs Gross Revenue', fontsize=14, fontweight='bold')
plt.legend()

corr = gross_df['IMDB_Rating'].corr(gross_df['Gross'])
plt.annotate(f'Correlation: {corr:.3f}', xy=(0.05, 0.90),
             xycoords='axes fraction', fontsize=12,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))

plt.tight_layout()
plt.savefig('rating_vs_gross.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'\n📊 Correlation between IMDb Rating and Gross Revenue: {corr:.4f}')

## 🎬 Step 9: Top Directors

In [ ]:
top_directors = df.groupby('Director').agg(
    Movie_Count=('Series_Title', 'count'),
    Avg_Rating=('IMDB_Rating', 'mean')
).reset_index()

# Filter directors with at least 3 movies
top_directors = top_directors[top_directors['Movie_Count'] >= 3].sort_values('Avg_Rating', ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By Movie Count
top_by_count = top_directors.sort_values('Movie_Count', ascending=True)
axes[0].barh(top_by_count['Director'], top_by_count['Movie_Count'],
             color=sns.color_palette('Blues_r', len(top_by_count)))
axes[0].set_title('🎬 Directors by Movie Count (≥3 films)', fontweight='bold')
axes[0].set_xlabel('Number of Movies')

# By Avg Rating
top_by_rating = top_directors.sort_values('Avg_Rating', ascending=True)
axes[1].barh(top_by_rating['Director'], top_by_rating['Avg_Rating'],
             color=sns.color_palette('Oranges_r', len(top_by_rating)))
axes[1].set_title('⭐ Directors by Average Rating (≥3 films)', fontweight='bold')
axes[1].set_xlabel('Average IMDb Rating')
axes[1].set_xlim(7.5, 9.5)

plt.tight_layout()
plt.savefig('top_directors.png', bbox_inches='tight', dpi=150)
plt.show()

## ⏱️ Step 10: Runtime Analysis

In [ ]:
runtime_df = df[['Series_Title', 'Runtime', 'IMDB_Rating']].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Runtime Distribution
axes[0].hist(runtime_df['Runtime'], bins=25, color='#2ca02c', edgecolor='black')
axes[0].axvline(runtime_df['Runtime'].mean(), color='red', linestyle='--',
                label=f'Mean: {runtime_df["Runtime"].mean():.0f} min')
axes[0].set_title('⏱️ Movie Runtime Distribution', fontweight='bold')
axes[0].set_xlabel('Runtime (minutes)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Rating vs Runtime
axes[1].scatter(runtime_df['Runtime'], runtime_df['IMDB_Rating'],
                alpha=0.5, color='#9467bd', edgecolors='grey', s=50)
axes[1].set_title('Runtime vs IMDb Rating', fontweight='bold')
axes[1].set_xlabel('Runtime (minutes)')
axes[1].set_ylabel('IMDb Rating')

corr = runtime_df['Runtime'].corr(runtime_df['IMDB_Rating'])
axes[1].annotate(f'Correlation: {corr:.3f}', xy=(0.05, 0.90),
                 xycoords='axes fraction', fontsize=11,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lavender', alpha=0.7))

plt.tight_layout()
plt.savefig('runtime_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 🔥 Step 11: Correlation Heatmap

In [ ]:
numeric_cols = ['IMDB_Rating', 'Meta_score', 'No_of_Votes', 'Gross', 'Runtime']
corr_df = df[numeric_cols].dropna()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('🔥 Correlation Heatmap of Movie Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

## 🏆 Step 12: Top 10 Movies by IMDb Rating

In [ ]:
top10 = df[['Series_Title', 'Released_Year', 'Director', 'IMDB_Rating', 'Genre', 'Gross']].sort_values(
    'IMDB_Rating', ascending=False).head(10).reset_index(drop=True)
top10.index += 1

print('🏆 TOP 10 IMDb MOVIES')
print('=' * 80)
top10

In [ ]:
plt.figure(figsize=(13, 5))
bars = plt.barh(top10['Series_Title'][::-1], top10['IMDB_Rating'][::-1],
                color=sns.color_palette('YlOrRd_r', 10))
plt.xlabel('IMDb Rating', fontsize=12)
plt.title('🏆 Top 10 Movies by IMDb Rating', fontsize=14, fontweight='bold')
plt.xlim(8.5, 9.5)

for bar, val in zip(bars, top10['IMDB_Rating'][::-1]):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('top10_movies.png', bbox_inches='tight', dpi=150)
plt.show()

## 📋 Step 13: Summary & Key Insights

In [ ]:
print('=' * 60)
print('📊 IMDb TOP 1000 MOVIES — KEY INSIGHTS')
print('=' * 60)
print(f'\n🎬 Total Movies Analyzed       : {len(df)}')
print(f'⭐ Average IMDb Rating         : {df["IMDB_Rating"].mean():.2f}')
print(f'⭐ Highest IMDb Rating         : {df["IMDB_Rating"].max()} — {df.loc[df["IMDB_Rating"].idxmax(), "Series_Title"]}')
print(f'📅 Year Range                  : {int(df["Released_Year"].min())} – {int(df["Released_Year"].max())}')

if 'Gross' in df.columns and df['Gross'].notna().sum() > 0:
    top_gross_movie = df.loc[df['Gross'].idxmax(), 'Series_Title']
    top_gross_val = df['Gross'].max() / 1e6
    print(f'💰 Highest Grossing Movie      : {top_gross_movie} (${top_gross_val:.0f}M)')

top_dir = df['Director'].value_counts().idxmax()
top_dir_count = df['Director'].value_counts().max()
print(f'🎬 Most Featured Director      : {top_dir} ({top_dir_count} movies)')

top_genre = genre_df['Genre'].value_counts().idxmax()
print(f'🎭 Most Common Genre           : {top_genre}')

print(f'\n✅ Analysis Complete!')